In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.bronze")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
INDICATOR_LABELS = {
    "euribor_3m": "EURIBOR 3-Month",
    "euribor_6m": "EURIBOR 6-Month",
}

mapping_expr = F.create_map([F.lit(x) for pair in INDICATOR_LABELS.items() for x in pair])

df = (
    spark.table("stocks.stage.macro_rates_raw")
    .withColumn("date",            F.to_date("date"))
    .withColumn("indicator_label", mapping_expr[F.col("indicator")])
    .withColumn("ingested_at",     F.current_timestamp())
)

In [ ]:
tbl = "stocks.bronze.macro_rates"
if not spark.catalog.tableExists(tbl):
    df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    df.alias("s"),
    "s.date = t.date AND s.indicator = t.indicator"
).whenNotMatchedInsertAll().execute()

for indicator in ["euribor_3m", "euribor_6m"]:
    print(f"\n--- {indicator} ---")
    spark.table(tbl).filter(f"indicator = '{indicator}'").orderBy("date").show(5, truncate=False)